Блок 1: Импорт

In [8]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.stats import chi2_contingency
import warnings
warnings.filterwarnings('ignore')

Блок 2: Загрузка данных

In [9]:
df = pd.read_csv('cookie_cats.txt', sep='\t')
print(f"Rows: {df.shape[0]}, Columns: {df.shape[1]}")

FileNotFoundError: [Errno 2] No such file or directory: 'cookie_cats.txt'

Блок 3: Проверка структуры


In [ ]:
print(df.head())
print(df.info())
print(df.describe())

Блок 4: Переименование колонок (если нужно)


In [ ]:
bins = [0, 1, 5, 10, 20, 30, 40, 50, 100, 200, 500, np.inf]
labels = ['0', '1-5', '6-10', '11-20', '21-30', '31-40', '41-50', '51-100', '101-200', '201-500', '500+']

df['level_group'] = pd.cut(df['sum_gamerounds'], bins=bins, labels=labels, right=False)

funnel = df['level_group'].value_counts().sort_index()
funnel_pct = (funnel / len(df) * 100).round(1)

funnel_df = pd.DataFrame({'users': funnel, 'percent': funnel_pct})
print(funnel_df)

Блок 5: Воронка


In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
bars = ax.bar(range(len(funnel_df)), funnel_df['users'], color='steelblue')

for i, row in enumerate(funnel_df.itertuples()):
    ax.text(i, row.users + max(funnel_df['users']) * 0.01, f"{row.percent}%", 
            ha='center', va='bottom', fontsize=9)

ax.set_xticks(range(len(funnel_df)))
ax.set_xticklabels(funnel_df.index, rotation=45)
ax.set_ylabel('Users')
ax.set_title('Funnel')

plt.tight_layout()
plt.show()

Блок 6: График воронки


In [ ]:
total = len(df)
zero_users = df[df['sum_gamerounds'] == 0].shape[0]
early_users = df[df['sum_gamerounds'] <= 5].shape[0]

print(f"Total users: {total:,}")
print(f"0 levels: {zero_users:,} ({zero_users/total*100:.1f}%)")
print(f"1-5 levels: {early_users - zero_users:,} ({(early_users - zero_users)/total*100:.1f}%)")
print(f"Dropoff by level 5: {early_users:,} ({early_users/total*100:.1f}%)")

Блок 7: Отток


In [ ]:
retention = df.groupby('version').agg({
    'retention_1': ['mean', 'count'],
    'retention_7': ['mean', 'count']
}).round(4)

retention.columns = ['ret_1_mean', 'ret_1_count', 'ret_7_mean', 'ret_7_count']
retention['ret_1_pct'] = (retention['ret_1_mean'] * 100).round(1)
retention['ret_7_pct'] = (retention['ret_7_mean'] * 100).round(1)

print(retention[['ret_1_pct', 'ret_7_pct']])

Блок 8: Retention по группам


Блок 9: T-test


In [ ]:
gate_30 = df[df['version'] == 'gate_30']
gate_40 = df[df['version'] == 'gate_40']

t_stat_1, p_value_1 = stats.ttest_ind(gate_30['retention_1'], gate_40['retention_1'])
t_stat_7, p_value_7 = stats.ttest_ind(gate_30['retention_7'], gate_40['retention_7'])

print(f"Retention Day 1: t={t_stat_1:.4f}, p={p_value_1:.4f}")
print(f"Retention Day 7: t={t_stat_7:.4f}, p={p_value_7:.4f}")
print(f"Significant (p<0.05): Day1={p_value_1<0.05}, Day7={p_value_7<0.05}")

Блок 10: SRM-проверка


In [ ]:
counts = df['version'].value_counts()
expected = len(df) / len(counts)

chi2, p_value = chi2_contingency([counts.values, [expected] * len(counts)])[:2]

print("Group distribution:")
for group, count in counts.items():
    print(f"  {group}: {count:,} ({count/len(df)*100:.1f}%)")
print(f"SRM p-value: {p_value:.4f}")
print(f"SRM check: {'PASS' if p_value > 0.05 else 'FAIL'}")

Блок 11: График retention


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for idx, col in enumerate(['retention_1', 'retention_7']):
    ax = axes[idx]
    data = df.groupby('version')[col].mean() * 100
    bars = ax.bar(data.index, data.values, color=['#2ecc71', '#3498db'])
    ax.set_title(f'{col} (%)')
    ax.set_ylim(0, 100)
    ax.grid(axis='y', alpha=0.3)
    
    for bar in bars:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height + 1, 
                f'{height:.1f}%', ha='center', va='bottom')

plt.tight_layout()
plt.show()

Блок 12: Итоговые метрики


In [ ]:
print("=" * 50)
print("PRODUCT METRICS")
print("=" * 50)

total = len(df)
ret_1 = df['retention_1'].mean() * 100
ret_7 = df['retention_7'].mean() * 100
median_rounds = df['sum_gamerounds'].median()
zero_pct = df[df['sum_gamerounds'] == 0].shape[0] / total * 100
early_pct = df[df['sum_gamerounds'] <= 5].shape[0] / total * 100

print(f"Total users: {total:,}")
print(f"Retention Day 1: {ret_1:.1f}%")
print(f"Retention Day 7: {ret_7:.1f}%")
print(f"Median rounds: {median_rounds}")
print(f"0 rounds: {zero_pct:.1f}%")
print(f"Dropoff by level 5: {early_pct:.1f}%")
print("=" * 50)

Блок 13: Экспорт


In [ ]:
df.to_csv('processed_data.csv', index=False)
print("Data exported")